# V14 Self-Play (Colab L4)

**Goal:** generate 200 V14 self-play games as the steady-state corpus for the pillar3c distillation step. Crisis mining (run separately on M5) contributes the failure-mode coverage.

**Player:** `pillar3b_epoch_20 + value_head_v14 + q_weight=2.0` — current SOA. Pillar3b's 1000-seed OOS eval: policy mean 17,255, P50 12,567, floor 2.5%. With matched value_head_v14, MCTS expected to lift further (per HISTORY 158 pattern, +30% MCTS over policy alone).

**Settings (V14 vs V13):**
- **400 sims** (matches V13 — pillar3b is stronger so we keep the same search budget)
- q_weight=2.0 (calibrated operating point — HISTORY 161, project_q_sweep_pillar3a memory)
- batch_size=8 (HISTORY 115)
- **max-turns=10000** (focus on floor not high tail — median is already decent)
- **temperature-moves=10** (reduced from V13's 15 — RNG already provides spawn diversity, and the early exploration moves looked "strange". V15 may go to 5.)
- Dirichlet α=0.3, weight=0.25 (defaults)

**Why 200 seeds (vs V13's 1000):** ChatGPT-recommended sequencing — start with a manageable selfplay batch to validate the pillar3b+v14 teacher generates good data, then either scale up to V14b (more selfplay) or move to pillar3c training on V14 + crisis_v14.

**Priority for this iteration: raise P1-P25 of policy-only outcomes.** Median is already strong. The Phase C finding (divergence emerges at turn 100-150) suggests floor failures come from mid-game density drift, not high-tail headroom. cap=10K is plenty.

## Upload to Drive (`MyDrive/alphatrain/`)

| file | size | source |
|---|---|---|
| `colorlines_path_b.tar.gz` | ~1.2 MB | local: `colorlines_path_b.tar.gz` |
| `pillar3b_epoch_20.pt` | ~136 MB | local: `alphatrain/data/pillar3b_epoch_20.pt` (already on Drive from pillar3b training) |
| `value_head_v14.pt` | ~38 KB | from Colab training notebook (`value_head_v14_qsweep_colab.ipynb`) |

## Seed ranges (disjoint from existing data)

- V13 selfplay used 1300000-1300999
- V13 crisis used 800000-808000 (M5)
- **V14 selfplay**: 1500000-1500199 (200 seeds)
- V14 crisis (M5, separate): 1600000-160NNN

## After this notebook

1. M5 generates V14 crisis in parallel (~6-12h)
2. Build V14 tensor: `build_expert_v2_tensor --games-dir data/selfplay_v14 data/crisis_v14 --policy-only-data --output alphatrain/data/v14_pillar3b.pt`
3. Train pillar3c with --target-temperature 0.5 (default) or tuned per V14 entropy check

Phase C divergence finding (turns 100-150 mid-game): consider **dense-state oversampling** for pillar3c training. Implementation: post-process V14 tensor to upweight states from low-final-score trajectories. Details TBD.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_path_b.tar.gz /content/
!cd /content && tar xzf colorlines_path_b.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
!cp {DRIVE}/pillar3b_epoch_20.pt /content/alphatrain/data/pillar3b.pt
!cp {DRIVE}/value_head_v14.pt /content/alphatrain/data/value_head_v14.pt
print(f'Model: {os.path.getsize("/content/alphatrain/data/pillar3b.pt")/1e6:.0f} MB')
print(f'Value head: {os.path.getsize("/content/alphatrain/data/value_head_v14.pt")/1e3:.0f} KB')

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

In [ ]:
# === V14 Self-Play ===
# Split across multiple Colab instances by editing SEED_START/SEED_END.
# V14 crisis (M5 local) will use 1600000+ — these ranges are disjoint.
SEED_START = 1500000
SEED_END = 1500200       # 200 games
# =======================

SIMS = 400               # same as V13
Q_WEIGHT = 2.0           # calibrated for value_head_v14 (matched-pair with pillar3b backbone)
WORKERS = 24             # Colab CPU
BATCH_SIZE = 8           # HISTORY 115
MAX_TURNS = 10000        # focus on floor not high tail; pillar3b median already 12.5K
TEMP_MOVES = 10          # reduced from V13's 15 — RNG provides spawn diversity
SAVE_DIR = f'{DRIVE}/selfplay_v14'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Seeds: {SEED_START}-{SEED_END-1} ({SEED_END-SEED_START} games)')
print(f'Sims: {SIMS}, q_weight: {Q_WEIGHT}, cap: {MAX_TURNS} turns, temp_moves: {TEMP_MOVES}')
print(f'Workers: {WORKERS}, bs: {BATCH_SIZE}')
print(f'Save: {SAVE_DIR}')

!python -m alphatrain.scripts.selfplay \
    --model /content/alphatrain/data/pillar3b.pt \
    --value-head-path /content/alphatrain/data/value_head_v14.pt \
    --q-weight {Q_WEIGHT} \
    --seed-start {SEED_START} --seed-end {SEED_END} \
    --sims {SIMS} --batch-size {BATCH_SIZE} \
    --save-dir {SAVE_DIR} \
    --workers {WORKERS} --device cuda \
    --temperature-moves {TEMP_MOVES} \
    --max-turns {MAX_TURNS} \
    --compile

## Notes

- **Resume support:** if Colab disconnects, just re-run the previous cell. It skips seeds whose game files already exist in `SAVE_DIR`.
- **Wall time on L4:** pillar3b at MCTS@400 with cap=15K. Stronger games take longer — expect ~2-5min per game in 24-worker parallel. **200 games ≈ 4-12h on L4.** If runtime exceeds Colab's 24h kill, split SEED_END to 1500100 (100 games), run, then a second instance for 1500100-1500199.
- **Watch:** GPU server prints `[GPU] X evals, Y fwd (avg bs=Z, ZZZZ evals/s)` every 10K forwards. avg bs should be ≥50 and evals/s in the thousands. If avg bs drops below 30, workers are starving — usually fine but worth flagging.
- **Output:** each game saved as `game_seed{N}_score{S}.json` in the save dir.
- **Crisis mining (M5 in parallel):**
  ```
  caffeinate -dim python -m alphatrain.scripts.crisis_mining \
      --model alphatrain/data/pillar3b_epoch_20.pt \
      --value-head-path alphatrain/data/value_head_v14.pt \
      --q-weight 2.0 \
      --seed-start 1600000 --seed-end 1601000 \
      --recovery-turns 15 --recovery-sims 600 \
      --prevention-turns 30 --prevention-sims 400 \
      --continue-turns 500 \
      --policy-max-turns 12000 \
      --workers 16 --device mps --batch-size 8 \
      --max-turns 25000 \
      --save-dir data/crisis_v14
  ```

## After both run

Build V14 tensor (M5, ~30 min):
```
python -m alphatrain.scripts.build_expert_v2_tensor \
    --games-dir data/selfplay_v14 data/crisis_v14 \
    --policy-only-data \
    --output alphatrain/data/v14_pillar3b.pt
```

Then train pillar3c (Colab, ~12h):
```
python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/v14_pillar3b.pt --amp --compile \
    --resume alphatrain/data/pillar3b_epoch_20.pt --warm-start \
    --epochs 17 --batch-size 32768 --lr 3e-4 --warmup-epochs 1 \
    --target-temperature 0.5
```

Decision criterion: pillar3c ≥ +15-20% over pillar3b mean OR floor cut by ≥30%. (Mean lift gets harder iteration-over-iteration; floor improvement is the explicit V14 goal per Phase C findings.)